# Checking ordinal

Proportional-odds (ordinal logit) model from statsmodels on simulated data: 5 ordered categories, temperature as the only covariate.

$$P(Y \le k \mid \text{temp}) = \sigma(c_k - \beta \cdot \text{temp}), \quad k = 1, \dots, 4$$

One shared slope $\beta$ and $K-1 = 4$ cutpoints. No intercept — the cutpoints play that role.

## Simulate data

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.miscmodels.ordinal_model import OrderedModel

rng = np.random.default_rng(0)
n = 500

# Covariate: room temperature (°C)
temp = rng.normal(22, 2, n)

# Latent model: y* = beta * temp + eps, eps ~ Logistic; 4 cutpoints -> 5 categories
beta_true = 1.2
cuts_true = np.array([24.0, 25.5, 27.0, 28.5])
y_star = beta_true * temp + rng.logistic(size=n)
y = np.searchsorted(cuts_true, y_star)  # 0..4

labels = ["very low", "low", "medium", "high", "very high"]
df = pd.DataFrame({
    "temp": temp,
    "level": pd.Categorical.from_codes(y, categories=labels, ordered=True),
})
print(df["level"].value_counts().sort_index())
df.head()

## Fit

In [ ]:
mod = OrderedModel(df["level"], df[["temp"]], distr="logit")
res = mod.fit(method="bfgs", disp=False)
print(res.summary())

In [ ]:
# Summary shows first cut + log-increments; transform back to actual cutpoints
cuts_hat = mod.transform_threshold_params(res.params)[1:-1]
print("true cutpoints:     ", cuts_true)
print("estimated cutpoints:", cuts_hat.round(3))

## Predicted category probabilities

In [ ]:
grid = pd.DataFrame({"temp": [18, 20, 22, 24, 26]})
# np.asarray: predict returns its own 0..n index, which would reindex to NaN
probs = pd.DataFrame(np.asarray(res.predict(grid)), columns=labels, index=grid["temp"])
probs.round(3)